In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
def convert_roman(roman_str):
    """
    Doc String
    """
    if not roman_str:
        return None
    try:
        return roman.fromRoman(roman_str)
    except:
        return None
    
roman_udf = udf(convert_roman, IntegerType())

In [0]:
nat_dex_df = spark.table(f"{STAGING_DATABASE_PREFIX}.nat_dex")

national_pokedex_df = (
    nat_dex_df
    .withColumn("pokemon_entry", explode(col("pokemon_entries")))
    .select(
        col("pokemon_entry.entry_number").alias("pokedex_number"),
        col("pokemon_entry.pokemon_species.name").alias("pokemon_name"),
        col("pokemon_entry.pokemon_species.url").alias("species_url")
    )
)

national_pokedex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.national_pokedex")

In [0]:
species_df = spark.table(f"{STAGING_DATABASE_PREFIX}.species")

national_pokedex_species_df = (
    species_df
    .withColumn("generation_name", split(col("generation.name"), '-')[1])
    .withColumn("generation", roman_udf(species_df["generation_name"]))
    .select(
        col('nat_dex_pokedex_no'),
        col('nat_dex_pokemon_name'),
        col("id"),
        col("name"),
        col('capture_rate'),
        col('base_happiness'),
        col('is_baby'),
        col("is_legendary"),
        col("is_mythical"),
        col('hatch_counter'),
        col('has_gender_differences'),
        col('forms_switchable'),
        col('growth_rate.name').alias('growth_rate'),
        col('evolution_chain'),
        col('generation_name'),
        col('generation')
    )
)

national_pokedex_species_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.species")

In [0]:
varieties_df = spark.table(f"{STAGING_DATABASE_PREFIX}.varieties")

national_pokedex_varieties_df = (
    varieties_df
    .withColumn(
        "hp", 
        expr("filter(stats, x -> x.stat.name = 'hp')[0].base_stat")
    ).withColumn(
        "attack", 
        expr("filter(stats, x -> x.stat.name = 'attack')[0].base_stat")
    ).withColumn(
        "defence", 
        expr("filter(stats, x -> x.stat.name = 'defense')[0].base_stat")
    ).withColumn(
        "special_attack", 
        expr("filter(stats, x -> x.stat.name = 'special-attack')[0].base_stat")
    ).withColumn(
        "special_defence", 
        expr("filter(stats, x -> x.stat.name = 'special-defense')[0].base_stat")
    ).withColumn(
        "speed", 
        expr("filter(stats, x -> x.stat.name = 'speed')[0].base_stat")
    ).withColumn(
        "type_1", 
        expr("CASE WHEN size(filter(types, x -> x.slot = 1)) > 0 THEN filter(types, x -> x.slot = 1)[0].type.name ELSE NULL END")
    ).withColumn(
        "type_2", 
        expr("CASE WHEN size(filter(types, x -> x.slot = 2)) > 0 THEN filter(types, x -> x.slot = 2)[0].type.name ELSE NULL END")
    ).withColumn(
        "hidden_ability",
        expr("try_element_at(transform(filter(abilities, x -> x.is_hidden = true), x -> x.ability.name), 1)")
    ).withColumn(
        "abilities",
        expr("transform(filter(abilities, x -> x.is_hidden = false), x -> x.ability.name)")
    )
    .select(
        col('nat_dex_pokedex_no'),
        col('nat_dex_pokemon_name'),
        col("species_id"),
        col("id"),
        col("name"),
        col("is_default"),
        col("abilities"),
        col("hidden_ability"),
        col("hp"),
        col("attack"),
        col("defence"),
        col("special_attack"),
        col("special_defence"),
        col("speed"),
        col("type_1"),
        col("type_2")
    )
)

national_pokedex_varieties_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.varieties")

In [0]:
forms_df = spark.table(f"{STAGING_DATABASE_PREFIX}.forms")

national_pokedex_forms_df = (
    forms_df
    .select(
        col('nat_dex_pokedex_no'),
        col('nat_dex_pokemon_name'),
        col("species_id"),
        col("variety_id"),
        col("id"),
        col("name"),
        col("form_name"),
        col('form_order'),
        col('is_mega'),
        col('is_battle_only'),
        col("trigger_conditions")
    )
)


national_pokedex_forms_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.forms")

In [0]:
evo_chain = spark.table(f'{STAGING_DATABASE_PREFIX}.evo_chain')

evo_chain_transformed = (
    evo_chain.select(
        col('id').alias('evo_chain_id'),
        col("chain.species.name").alias('basic_stage'),
        col("chain.species.url").alias('basic_stage_species_url'),
        col('chain.evolves_to').alias('evolves_to_stage_1'),
        col("chain.is_baby").alias('is_baby')
    )
    .withColumn('stage_1_evolution', explode_outer(col('evolves_to_stage_1')))
    .withColumn('stage_1_details', explode_outer(col('stage_1_evolution.evolution_details')))
    .select(
        "evo_chain_id",
        coalesce(col('stage_1_details.base_form.name'), col('basic_stage')).alias('basic_stage_pokemon_name'),
        "basic_stage_species_url",
        coalesce(col('stage_1_details.evolved_form.name'), col("stage_1_evolution.species.name")).alias('stage_1_pokemon_name'),
        col("stage_1_evolution.species.url").alias('stage_1_species_url'),
        col('stage_1_evolution.evolves_to').alias('evolves_to_stage_2'),
        # col('stage_1_details')
    )
    .withColumn('stage_2_evolution', explode_outer(col('evolves_to_stage_2')))
    .withColumn('stage_2_details', explode_outer(col('stage_2_evolution.evolution_details')))
    .select(
        "evo_chain_id",
        "basic_stage_pokemon_name",
        "basic_stage_species_url",
        "stage_1_pokemon_name",
        "stage_1_species_url",
        coalesce(col('stage_2_details.evolved_form.name'), col("stage_2_evolution.species.name")).alias('stage_2_pokemon_name'),
        col("stage_2_evolution.species.url").alias('stage_2_species_url'),
        # col("stage_1_details"),
        # col('stage_2_details')
    )
    .dropDuplicates()
)

evo_chain_transformed.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.evolution_chain")